In [4]:
import pandas as pd

df = pd.read_csv("../../../etl/raw_data/한국고전종합DB_관계망/itkc_person_relations.csv")


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 206764 entries, 0 to 206763
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   person_id            206764 non-null  str  
 1   person_name          206764 non-null  str  
 2   relation_type        206764 non-null  str  
 3   related_person_id    206764 non-null  str  
 4   related_person_name  206764 non-null  str  
 5   related_birth_year   113597 non-null  str  
 6   related_death_year   93605 non-null   str  
 7   related_bonkwan      190180 non-null  str  
 8   related_father       167378 non-null  str  
 9   related_count        206764 non-null  str  
 10  evidence_url         206683 non-null  str  
 11  detail_url           206764 non-null  str  
dtypes: str(12)
memory usage: 18.9 MB


In [8]:
df.tail(2)

,person_id,person_name,relation_type,related_person_id,related_person_name,related_birth_year,related_death_year,related_bonkwan,related_father,related_count,evidence_url,detail_url
206762,P059096,징엄(澄儼),제자,P058757,종린(宗璘),1127,1179,개성(開城),왕해(王楷),9명,https://encykorea.aks.ac.kr/Article/E0052925,https://db.itkc.or.kr/people/view?gubun=person...
206763,P059097,징천(澄泉),스승,P064320,혜근(惠勤),1320,1376,NaN,아서구(牙瑞具),12명,http://thesaurus.itkc.or.kr/search/view?dataId...,https://db.itkc.or.kr/people/view?gubun=person...


In [9]:
df['person_id'].nunique()

56212

In [10]:
df['relation_type'].unique()

<StringArray>
[ '교유',  '스승',  '제자',   '부', '증조부',  '조부',  '형제',   '자',  '장인',  '사위',  '생부',
  '남편',  '출자',  '아내',   '모',  '생모']
Length: 16, dtype: str

In [11]:
df[df['relation_type'] == '부'].head()

,person_id,person_name,relation_type,related_person_id,related_person_name,related_birth_year,related_death_year,related_bonkwan,related_father,related_count,evidence_url,detail_url
14,P000005,각안(覺岸),부,P061153,최철(崔徹),NaN,NaN,경주(慶州),NaN,1명,https://encykorea.aks.ac.kr/Article/E0000463,https://db.itkc.or.kr/people/view?gubun=person...
35,P000012,각해(覺海),부,P046563,이진(李瑱),1244,1321,경주(慶州),이핵(李翮),10명,http://thesaurus.itkc.or.kr/search/view?dataId...,https://db.itkc.or.kr/people/view?gubun=person...
41,P000014,감경인(甘景仁),부,P000017,감예종(甘禮從),NaN,NaN,회산(檜山),NaN,2명,https://encykorea.aks.ac.kr/Article/E0000689,https://db.itkc.or.kr/people/view?gubun=person...
44,P000015,감기현(甘麒鉉),부,P000019,감재원(甘在元),1849,1920,회산(檜山),감회정(甘檜廷),4명,http://people.aks.ac.kr/front/dirSer/ppl/pplVi...,https://db.itkc.or.kr/people/view?gubun=person...
49,P000019,감재원(甘在元),부,P000020,감회정(甘檜廷),1759,1816,회산(檜山),감수운(甘守雲),2명,http://people.aks.ac.kr/front/dirSer/ppl/pplVi...,https://db.itkc.or.kr/people/view?gubun=person...


## EDA 정리: `person_relations.csv` 인물 관계 데이터

현재 파일은 `itkc_person_relations.csv`이며, 한 행은 인물 1명의 속성이 아니라 **인물과 인물 사이의 관계 1개**로 본다.

```text
person_id 인물 -- relation_type --> related_person_id 인물
```

예를 들어 `P000002 각겸(覺謙)`이 여러 명과 `교유` 관계를 가지면, 이것을 한 row에 리스트로 묶지 않는다. Neo4j에서는 관계 여러 개로 표현하는 것이 자연스럽다.

```text
P000002 각겸 --교유--> P002331 권근
P000002 각겸 --교유--> P020378 석나암
P000002 각겸 --교유--> P039883 이색
```

따라서 이 데이터는 `Person` 노드 생성용이라기보다 `Person - RELATED_TO - Person` 관계 생성용으로 먼저 해석한다.

### 1. 컬럼별 의미와 사용 판단

| 컬럼 | 의미 | 처리 | 사용 방식/이유 |
|---|---|---|---|
| `person_id` | 시작 인물 ID | 사용 | 관계의 source Person. |
| `person_name` | 시작 인물 이름 | 보존/검수 | `person_id`의 이름 검수용. 정식 인물 속성은 `itkc_people.csv`와 비교한다. |
| `relation_type` | 관계 유형 | 사용 | `교유`, `부` 같은 관계 종류. 관계 타입 또는 관계 속성으로 사용한다. |
| `related_person_id` | 대상 인물 ID | 사용 | 관계의 target Person. |
| `related_person_name` | 대상 인물 이름 | 보존/검수 | `related_person_id`의 이름 검수용. |
| `related_birth_year` | 대상 인물 출생연도 | Person 보강 후보 | 관계 속성이 아니라 대상 Person의 속성 후보. |
| `related_death_year` | 대상 인물 사망연도 | Person 보강 후보 | 관계 속성이 아니라 대상 Person의 속성 후보. |
| `related_bonkwan` | 대상 인물 본관 | Person 보강 후보 | 관계 속성이 아니라 대상 Person의 속성 후보. |
| `related_father` | 대상 인물의 아버지 | Person 보강 후보 | 관계 속성이 아니라 대상 Person의 가족 정보 후보. |
| `related_count` | 관련 수 표시 | 선택 보존 | 의미가 관계 강도인지 대상 인물의 관련 수인지 애매하므로 핵심 관계 판단에는 쓰지 않는다. |
| `evidence_url` | 관계 근거 URL | 보존 | 관계의 근거 출처로 사용한다. RAG 근거 수집에도 활용 가능하다. |
| `detail_url` | 시작 인물 상세 URL | 보존 | Person 상세 출처 URL로 사용한다. |

### 2. 중복 제거 기준

관계 데이터에서는 같은 사람이 여러 명과 연결되는 것이 정상이다. 따라서 `person_id`만 기준으로 중복 제거하면 안 된다.

1차 중복 제거 기준은 다음처럼 잡는다.

```python
person_relation_df = df.drop_duplicates(
    subset=["person_id", "related_person_id", "relation_type"]
)
```

이 기준은 같은 시작 인물, 같은 대상 인물, 같은 관계 유형이 모두 같을 때만 중복으로 본다.

### 3. 관계 import용 컬럼

Neo4j 관계 CSV만 따로 만든다면 최소 컬럼은 다음 정도면 된다.

```python
person_relation_edges = person_relation_df[
    ["person_id", "related_person_id", "relation_type", "evidence_url"]
]
```

출처와 원본 값을 조금 더 보존하려면 다음처럼 가져간다.

```python
person_relation_edges = person_relation_df[
    [
        "person_id",
        "related_person_id",
        "relation_type",
        "related_count",
        "evidence_url",
        "detail_url",
    ]
]
```

`person_name`, `related_person_name`, `related_birth_year`, `related_death_year`, `related_bonkwan`, `related_father`는 관계 import용에서는 제외할 수 있다. 다만 버리는 것이 아니라 Person 노드 보강용 staging 데이터로 따로 볼 수 있다.

### 4. `relation_type='부'` 해석

`relation_type`이 `부`인 행은 보통 다음처럼 해석한다.

```text
person_id 인물의 아버지 = related_person_id 인물
```

예를 들어 `각안(覺岸)`의 `부`가 `최철(崔徹)`로 나와도 바로 오류로 보지 않는다. `각안`은 성명이 아니라 법명/승려명일 수 있기 때문에 겉으로 보이는 성이 다를 수 있다.

따라서 성씨 불일치만으로 관계를 제거하지 않고, 검수 후보 정도로만 본다.

### 5. URL과 RAG 활용

`evidence_url`, `detail_url`은 그래프 구조 자체에는 필수는 아니지만, RAG와 검증 가능성을 위해 보존하는 것이 좋다.

| URL 컬럼 | 사용 방향 |
|---|---|
| `evidence_url` | 인물 관계의 근거 URL. 관계 속성 또는 Evidence 노드 후보. |
| `detail_url` | 시작 인물 상세 페이지 URL. Person 노드의 source URL 후보. |

Tavily를 쓰면 이 URL들의 본문을 추출해서 RAG 근거로 사용할 수 있다. 다만 Tavily 결과를 바로 확정 관계로 넣기보다, URL 본문 추출과 답변 근거 보강용으로 쓰는 것이 안전하다.

```text
Neo4j = 인물/사건/용어의 구조적 관계
Vector RAG = 설명문, URL 본문, term_desc 검색
Tavily = URL 본문 추출과 외부 웹 근거 보강
```

따라서 이 프로젝트의 RAG 구조는 `Graph RAG + Vector RAG + Web RAG`를 섞은 Hybrid RAG로 볼 수 있다.

### 6. 1차 결론

- 한 행은 Person 노드 1개가 아니라 Person-Person 관계 1개다.
- 한 사람의 관련 인물을 한 row에 리스트로 묶지 않는다.
- 중복 제거는 `person_id`, `related_person_id`, `relation_type` 기준으로 한다.
- 관계 import용에서는 `person_name`, `related_person_name`, `related_birth_year`, `related_death_year`, `related_bonkwan`, `related_father`를 제외할 수 있다.
- 제외한 인물 속성 컬럼은 버리는 것이 아니라 Person 노드 보강용 staging으로 남긴다.
- `related_count`는 의미가 애매하므로 핵심 관계 판단에는 쓰지 않고 선택 보존한다.
- `evidence_url`, `detail_url`은 RAG와 출처 추적을 위해 보존한다.
